# Super-Resolution Mapping from Medium-Resolution Satellite Imagery (Sentinel-2 10m → 2.5m)
## Theoretical Analysis, Multi-Spectral Loss Design (NDVI, NDWI, NDBI), and Fine-Tuning Methodology

**Department of Computer Science & Engineering | Graphic Era Deemed to be University, Dehradun**  
**Project Team ID:** MP2026CSE170  
**Faculty Guide:** Mr. Vishnu Singh Shekhawat  
**Students:** Aaditya Singh Rawat, Anmol Rawat, Dhriti Rana, Ayush Singh Rauthan  

---

## 1. Executive Summary & Project Objective

Sentinel-2 provides freely accessible multispectral Earth observation with 5-day revisit cycles. However, its native spatial resolution of **10 meters** (for RGB and NIR bands) is insufficient for fine-scale remote sensing applications such as:
- Small building identification and footprint delineation
- Narrow road network extraction
- Agricultural field boundary mapping in fragmented landscapes
- Sub-pixel vegetation monitoring in heterogeneous environments (e.g., Dehradun and Indian sub-regions)

### Primary Goal
Enhance 10m Sentinel-2 multispectral imagery by **4× super-resolution** to **2.5 meters/pixel**, while maintaining strict **radiometric and spectral fidelity** across critical vegetation, water, and built-up indices (**NDVI, NDWI, NDBI**).

## 2. In-Depth Study of Baseline SEN2SR (Lite vs. Full Models)

The **SEN2SR** framework (Aybar et al., 2024/2026; *ESA OpenSR*) addresses Sentinel-2 super-resolution through a 3-stage modular decomposition:

```text
[10m RGBN (B02,B03,B04,B08)] ──────────Stage 1 (4x)──────────► [2.5m RGBN]
                                                                  │ (spatial guidance)
[20m RSWIR (B05,B06,B07,B8A,B11,B12)] ─Stage 2 (2x)─► [10m RSWIR] ──Stage 3 (4x)─► [2.5m RSWIR]
```

### 2.1 Model Architecture Breakdown

| Component | SEN2SR Lite Architecture | SEN2SR Full Architecture |
| :--- | :--- | :--- |
| **Stage 1: RGBN 10m → 2.5m (`sr_model`)** | `CNNSR`: 6 residual blocks, 24 feature channels, pixel-shuffle | `MambaSR`: 6 Vision State-Space blocks (`embed_dim=96`, 8 heads, selective SSM scan) |
| **Stage 2: 20m → 10m Guidance (`f2_model`)** | `CNNSR`: 10 in-channels, 6 out-channels, 24 features | `Swin2SR`: Swin Transformer V2 (`window_size=4`, `embed_dim=192`, 8 heads) |
| **Stage 3: 10m → 2.5m RSWIR (`model`)** | `CNNSR`: 10 in-channels, 6 out-channels, 24 features | `Swin2SR`: Swin Transformer V2 (`window_size=8`, `embed_dim=120`, 8 heads) |
| **Hardware Requirements** | Runs on any standard GPU / CPU (pure PyTorch) | Requires CUDA kernel compilation (`mamba-ssm`, `triton`), high VRAM |
| **Inference Speed** | Very fast (seconds per tile) | Heavy and memory-intensive |

## 3. Empirical Limitations of Current Baseline Models

Testing the out-of-the-box pretrained checkpoints reveals two significant challenges:

### Issue A: Spatial Softness / Blurring on 4× Zoom
- Pretrained models were trained using basic **$L_1$ pixel regression loss**.
- $L_1$ penalizes large deviations, which causes the network to predict the statistical mean (blurring) whenever it is uncertain about fine spatial details (edges of buildings, narrow pathways, parcel boundaries).
- In the Lite model, having only 24 feature channels means the model lacks capacity to reconstruct high-frequency spatial gradients without explicit edge supervision.

### Issue B: Subtle / Negligible Spectral Index Differences
- Both models enforce a mathematical post-processing constraint called **`HardConstraint`** (`sr_hard_constraint.safetensor`).
- `HardConstraint` replaces the low frequencies of the super-resolved output with the exact low frequencies of the 10m input using a low-pass Fourier/Gaussian mask:
  $$\mathcal{D}(\hat{X}_{\text{SR}}) \equiv X_{\text{LR}}$$
- Because the low-frequency envelope is mathematically fixed, the macro-average of NDVI/NDWI across a patch cannot deviate from the 10m baseline. Sub-pixel spectral contrast between vegetation and built-up land remains smudged without explicit spectral index loss.

## 4. Multi-Spectral Index Mathematical Formulations

Our project specifically evaluates and supervises three primary remote sensing spectral indices:

### 1. NDVI (Normalized Difference Vegetation Index)
$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red} + \epsilon} = \frac{B08 - B04}{B08 + B04 + \epsilon}$$
- **Target:** Biomass, crop health, field parcel borders.
- **Implementation:** Directly supervised in 4-band fine-tuning.

### 2. NDWI (Normalized Difference Water Index — McFeeters 1996)
$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR} + \epsilon} = \frac{B03 - B08}{B03 + B08 + \epsilon}$$
- **Target:** Water bodies, ponds, river channels, and moisture edges.
- **Implementation:** Directly supervised in 4-band fine-tuning since both Green (B03) and NIR (B08) are in the 4-band RGBN stack.

### 3. NDBI (Normalized Difference Built-up Index — Zha et al. 2003)
$$\text{NDBI} = \frac{\text{SWIR} - \text{NIR}}{\text{SWIR} + \text{NIR} + \epsilon} = \frac{B11 - B08}{B11 + B08 + \epsilon}$$
- **Target:** Concrete, urban settlements, buildings, roads, and impervious surfaces.
- **Sensor Constraint & Role:** SWIR (B11) is a satellite-specific 20m band not captured by 4-band airborne NAIP cameras. Therefore:
  1. In **4-Band Fine-Tuning**, built-up features (buildings/roads) are sharpened geometrically using the **Edge-Aware Sobel Loss** ($\|\nabla \hat{Y} - \nabla Y\|_1$).
  2. In **10-Band Full Model Inference**, all 10 bands (including B11 SWIR) are super-resolved, and NDBI is quantitatively evaluated across urban zones.

## 5. Proposed Fine-Tuning Methodology: Compound Loss Function

As formulated in **Chapters 3 & 4 (Figs 4.2 & 4.3)** of our Major Project Synopsis, our engineering contribution is to fine-tune the pretrained baseline using a **Multi-Objective Compound Loss Function**:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{pixel}} + \lambda_{\text{edge}} \cdot \mathcal{L}_{\text{edge}} + \lambda_{\text{spectral}} \cdot \mathcal{L}_{\text{spectral}}$$

Where:
- $\mathcal{L}_{\text{pixel}} = \|\hat{Y} - Y_{\text{GT}}\|_1$ (Radiometric reflectance anchor)
- $\mathcal{L}_{\text{edge}} = \|\nabla_x \hat{Y} - \nabla_x Y_{\text{GT}}\|_1 + \|\nabla_y \hat{Y} - \nabla_y Y_{\text{GT}}\|_1$ (Sobel edge gradient loss)
- $\mathcal{L}_{\text{spectral}} = \|\text{NDVI}_{\text{pred}} - \text{NDVI}_{\text{GT}}\|_1 + 0.5 \|\text{NDWI}_{\text{pred}} - \text{NDWI}_{\text{GT}}\|_1$ (Differentiable vegetation & water index loss)

In [ ]:
# Complete Multi-Spectral & Edge Compound Loss in PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiIndexSpectralLoss(nn.Module):
    """
    Differentiable loss supervising both NDVI (Vegetation) and NDWI (Water).
    Band Indices: 0:Blue (B02), 1:Green (B03), 2:Red (B04), 3:NIR (B08)
    """
    def __init__(self, eps=1e-6, weight_ndwi=0.5):
        super().__init__()
        self.eps = eps
        self.weight_ndwi = weight_ndwi

    def forward(self, pred, target):
        green_p, green_t = pred[:, 1], target[:, 1]
        red_p,   red_t   = pred[:, 2], target[:, 2]
        nir_p,   nir_t   = pred[:, 3], target[:, 3]

        # NDVI = (NIR - Red) / (NIR + Red)
        ndvi_p = (nir_p - red_p) / (nir_p + red_p + self.eps)
        ndvi_t = (nir_t - red_t) / (nir_t + red_t + self.eps)
        loss_ndvi = F.l1_loss(ndvi_p, ndvi_t)

        # NDWI = (Green - NIR) / (Green + NIR)
        ndwi_p = (green_p - nir_p) / (green_p + nir_p + self.eps)
        ndwi_t = (green_t - nir_t) / (green_t + nir_t + self.eps)
        loss_ndwi = F.l1_loss(ndwi_p, ndwi_t)

        return loss_ndvi + self.weight_ndwi * loss_ndwi, {'ndvi': loss_ndvi.item(), 'ndwi': loss_ndwi.item()}


class EdgeAwareSobelLoss(nn.Module):
    """
    Penalizes blurred gradients by matching spatial derivatives along X and Y axes.
    Directly sharpens building boundaries, roads, and parcel lines (NDBI proxy).
    """
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, pred, target):
        B, C, H, W = pred.shape
        loss = 0.0
        for c in range(C):
            p = pred[:, c:c+1]
            t = target[:, c:c+1]
            gx_p = F.conv2d(p, self.sobel_x, padding=1)
            gy_p = F.conv2d(p, self.sobel_y, padding=1)
            gx_t = F.conv2d(t, self.sobel_x, padding=1)
            gy_t = F.conv2d(t, self.sobel_y, padding=1)
            loss += F.l1_loss(gx_p, gx_t) + F.l1_loss(gy_p, gy_t)
        return loss / C


class MajorProjectCompoundLoss(nn.Module):
    def __init__(self, lambda_edge=0.20, lambda_spectral=0.15):
        super().__init__()
        self.lambda_edge = lambda_edge
        self.lambda_spectral = lambda_spectral
        self.l1 = nn.L1Loss()
        self.edge = EdgeAwareSobelLoss()
        self.spec = MultiIndexSpectralLoss()

    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_edge = self.edge(pred, target)
        loss_spec, spec_details = self.spec(pred, target)
        total = loss_l1 + (self.lambda_edge * loss_edge) + (self.lambda_spectral * loss_spec)
        return total, {'l1': loss_l1.item(), 'edge': loss_edge.item(), **spec_details}

## 6. Two-Phase Experimental Roadmap & Quantitative Metrics

1. **Phase 1: Prototyping on SEN2SR Lite (`CNNSR`)**
   - Fast training on paired patches from `sentinaiplow` and `sentinaiphigh` (~15 min on Colab T4).
   - Validates that adding Edge Loss + NDVI/NDWI Loss eliminates blurring and preserves water/vegetation fidelity.

2. **Phase 2: Scaling to SEN2SR Full (`MambaSR` + `Swin2SR`)**
   - Leverages Vision State-Space modeling for 4× RGBN super-resolution and Swin Transformers for SWIR bands.
   - Evaluates NDBI across built-up urban zones in Indian Sentinel-2 study areas (Dehradun / Graphic Era campus).

3. **Target Evaluation Metrics:**
   - **Spatial:** PSNR (dB), SSIM, RMSE.
   - **Spectral:** SAM (Spectral Angle Mapper), NDVI-MAE, NDWI-MAE, NDBI consistency.